In [3]:
import time
import re
import pandas as pd
import requests

BASE = "https://fbref.com/en/comps/9/{season}/schedule/{season}-Premier-League-Scores-and-Fixtures"

SESSION = requests.Session()

HEADERS = {
    # Un UA réaliste
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,fr-FR;q=0.8,fr;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Referer": "https://fbref.com/",
    "DNT": "1",
    "Upgrade-Insecure-Requests": "1",
}

def get_html(url: str, sleep_s: float = 3.0, max_retries: int = 3) -> str:
    """Télécharge une page HTML avec retries/backoff pour limiter les 403/429."""
    for attempt in range(max_retries):
        r = SESSION.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            time.sleep(sleep_s)  # pause polie
            return r.text

        # 403/429 => on ralentit et on retente
        if r.status_code in (403, 429):
            wait = sleep_s * (attempt + 1) * 2
            print(f"⚠️ {r.status_code} sur {url} — retry dans {wait:.1f}s")
            time.sleep(wait)
            continue

        r.raise_for_status()

    raise RuntimeError(f"Impossible d'accéder à {url} (dernier status: {r.status_code}).")

def extract_schedule_table(html: str) -> pd.DataFrame:
    """Extrait la table schedule (directement ou depuis commentaires HTML)."""
    # 1) essai direct
    try:
        tables = pd.read_html(html)
        if tables:
            return tables[0].copy()
    except Exception:
        pass

    # 2) fallback: tables dans commentaires
    comments = re.findall(r"<!--(.*?)-->", html, flags=re.DOTALL)
    for c in comments:
        try:
            t = pd.read_html(c)
            if t:
                return t[0].copy()
        except Exception:
            continue

    raise ValueError("Aucune table schedule trouvée sur la page.")

def fetch_fbref_schedule(season: str, sleep_s: float = 3.0) -> pd.DataFrame:
    url = BASE.format(season=season)
    html = get_html(url, sleep_s=sleep_s)

    df = extract_schedule_table(html)

    # nettoyage minimal
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    if "wk" in df.columns:
        df = df[df["wk"].notna()].copy()

    df["season"] = season
    return df

def fetch_all_seasons(start_year: int = 2015, end_year: int = 2025) -> pd.DataFrame:
    all_dfs = []
    for y in range(start_year, end_year + 1):
        season = f"{y}-{y+1}"
        print(f"Fetching {season}...")
        df = fetch_fbref_schedule(season, sleep_s=3.0)
        all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True)

# --- RUN ---
df_all = fetch_all_seasons(2015, 2025)

df_all.to_csv("fbref_pl_schedules_2015_2026.csv", index=False)
df_all.to_parquet("fbref_pl_schedules_2015_2026.parquet", index=False)

print("Done:", df_all.shape)
print(df_all.head())


Fetching 2015-2016...
⚠️ 403 sur https://fbref.com/en/comps/9/2015-2016/schedule/2015-2016-Premier-League-Scores-and-Fixtures — retry dans 6.0s
⚠️ 403 sur https://fbref.com/en/comps/9/2015-2016/schedule/2015-2016-Premier-League-Scores-and-Fixtures — retry dans 12.0s
⚠️ 403 sur https://fbref.com/en/comps/9/2015-2016/schedule/2015-2016-Premier-League-Scores-and-Fixtures — retry dans 18.0s


RuntimeError: Impossible d'accéder à https://fbref.com/en/comps/9/2015-2016/schedule/2015-2016-Premier-League-Scores-and-Fixtures (dernier status: 403).

In [151]:
from pathlib import Path
import pandas as pd
import re

def extract_schedule_from_file(path):
    b_html = path.read_text(encoding="utf-8", errors="ignore")

    try:
        tables = pd.read_html(b_html)
        if tables:
            return tables[0]
    except Exception:
        pass

    comments = re.findall(r"<!--(.*?)-->", html, flags=re.DOTALL)
    for c in comments:
        try:
            t = pd.read_html(c)
            if t:
                return t[0]
        except Exception:
            continue

    raise ValueError(f"Aucune table trouvée dans {path.name}")

dfs = []
for p in Path("../raw_html").glob("*.html"):
    df = extract_schedule_from_file(p)
    df["season"] = p.stem
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)


C:\Users\flere\AppData\Local\Temp\ipykernel_14768\2403097589.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(b_html)
C:\Users\flere\AppData\Local\Temp\ipykernel_14768\2403097589.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(b_html)
C:\Users\flere\AppData\Local\Temp\ipykernel_14768\2403097589.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(b_html)
C:\Users\flere\AppData\Local\Temp\ipykernel_14768\2403097589.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal 

##### from pathlib import Path

for p in Path("../raw_html").glob("*.html"):
    print(p)


In [8]:
import os
os.getcwd()


'C:\\Users\\flere\\Documents\\Projet Pyt\\notebooks'

In [140]:
df_all['season'].unique()

array(['2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020',
       '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025',
       '2025-2026'], dtype=object)

In [153]:
df = df_all.copy()

df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace(r"\s+", "_", regex=True)
      .str.replace(r"[^a-z0-9_]", "", regex=True)
)


In [154]:
df = df.drop(columns=["match_report", "notes"], errors="ignore")


In [155]:
df["wk"] = df["wk"].astype("Int64")

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["attendance"] = df["attendance"].astype("Int64")

In [156]:
df["score"] = (
    df["score"]
      .astype(str)
      .str.replace("–", "-", regex=False)
)


In [157]:
goals = df["score"].str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$")

df["home_goals"] = pd.to_numeric(goals[0], errors="coerce")
df["away_goals"] = pd.to_numeric(goals[1], errors="coerce")


In [158]:
df = df[df["home_goals"].notna()].copy()

In [159]:
df["season_start"] = df["season"].str[:4].astype(int)

In [160]:
df = df[
    [
        "season_start", "wk", "date", "time",
        "home", "away",
        "home_goals", "away_goals",
        "attendance", "venue", "referee"
    ]
].copy()


In [161]:
df

,season_start,wk,date,time,home,away,home_goals,away_goals,attendance,venue,referee
0,2015,1,2015-08-08,12:45,Manchester Utd,Tottenham Hotspur,1.0,0.0,75261,Old Trafford,Jonathan Moss
1,2015,1,2015-08-08,15:00,Leicester City,Sunderland,4.0,2.0,32242,King Power Stadium,Lee Mason
2,2015,1,2015-08-08,15:00,Bournemouth,Aston Villa,0.0,1.0,11155,Vitality Stadium,Mark Clattenburg
3,2015,1,2015-08-08,15:00,Everton,Watford,2.0,2.0,39063,Goodison Park,Mike Jones
4,2015,1,2015-08-08,15:00,Norwich City,Crystal Palace,1.0,3.0,27036,Carrow Road,Simon Hooper
...,...,...,...,...,...,...,...,...,...,...,...
4559,2025,26,2026-02-11,19:30,Nottingham Forest,Wolves,0.0,0.0,29921,The City Ground,Tim Robinson
4560,2025,26,2026-02-11,19:40,Crystal Palace,Burnley,2.0,3.0,24641,Selhurst Park,Farai Hallam
4561,2025,26,2026-02-11,20:15,Sunderland,Liverpool,0.0,1.0,47159,Stadium of Light,Chris Kavanagh
4562,2025,26,2026-02-12,20:00,Brentford,Arsenal,1.0,1.0,17224,Gtech Community Stadium,John Brooks


In [163]:
import pandas as pd

def build_team_matches(df_all: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df_all.copy()

    # 1) Noms de colonnes
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    # 2) Drop colonnes inutiles
    df = df.drop(columns=["match_report", "notes"], errors="ignore")

    # 3) Types
    df["wk"] = pd.to_numeric(df["wk"], errors="coerce").astype("Int64")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["attendance"] = pd.to_numeric(df["attendance"], errors="coerce").astype("Int64")

    # 4) Score -> home_goals / away_goals
    df["score"] = df["score"].astype(str).str.replace("–", "-", regex=False)
    goals = df["score"].str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$")
    df["home_goals"] = pd.to_numeric(goals[0], errors="coerce")
    df["away_goals"] = pd.to_numeric(goals[1], errors="coerce")

    # 5) Filtrer matchs joués uniquement
    df = df[df["home_goals"].notna()].copy()

    # 6) Saison (année de départ)
    # df_all a season type "2015-2016"
    df["season_start"] = df["season"].astype(str).str[:4].astype(int)

    # 7) Clé match (stable)
    df["match_id"] = (
        df["season_start"].astype(str) + "_" +
        df["date"].dt.strftime("%Y-%m-%d") + "_" +
        df["home"].astype(str) + "_" +
        df["away"].astype(str)
    )

    matches_clean = df[[
        "match_id", "season_start", "wk", "date",
        "home", "away",
        "home_goals", "away_goals",
        "attendance", "venue", "referee"
    ]].rename(columns={"wk": "matchweek"}).copy()

    # =========
    # 8) Passer en format long: 1 ligne = 1 équipe dans 1 match
    # =========
    home_df = matches_clean.rename(columns={
        "home": "team",
        "away": "opponent",
        "home_goals": "goals_for",
        "away_goals": "goals_against"
    }).copy()
    home_df["venue"] = "home"

    away_df = matches_clean.rename(columns={
        "away": "team",
        "home": "opponent",
        "away_goals": "goals_for",
        "home_goals": "goals_against"
    }).copy()
    away_df["venue"] = "away"

    tm = pd.concat([home_df, away_df], ignore_index=True)

    # 9) Points
    tm["points"] = (tm["goals_for"] > tm["goals_against"]).astype(int) * 3
    tm.loc[tm["goals_for"] == tm["goals_against"], "points"] = 1

    # 10) Clé journée façon toi
    tm["mw_key"] = "J" + tm["matchweek"].astype(int).astype(str) + "_" + tm["season_start"].astype(str)

    # 11) Trier + cumul points
    tm = tm.sort_values(["season_start", "team", "matchweek", "date"]).reset_index(drop=True)

    tm["cum_points_after"] = (
        tm.groupby(["season_start", "team"])["points"].cumsum()
    )
    tm["cum_points_before"] = (
        tm.groupby(["season_start", "team"])["cum_points_after"].shift(1).fillna(0)
    )

    # 12) Cum GF/GA/GD after
    tm["cum_gf_after"] = tm.groupby(["season_start", "team"])["goals_for"].cumsum()
    tm["cum_ga_after"] = tm.groupby(["season_start", "team"])["goals_against"].cumsum()
    tm["cum_gd_after"] = tm["cum_gf_after"] - tm["cum_ga_after"]

    # 13) Rank_after (sans apply, propre)
    tm = tm.sort_values(
        ["season_start", "matchweek", "cum_points_after", "cum_gd_after", "cum_gf_after", "team"],
        ascending=[True, True, False, False, False, True]
    ).reset_index(drop=True)

    tm["rank_after"] = tm.groupby(["season_start", "matchweek"]).cumcount() + 1

    # 14) Rank_before
    tm = tm.sort_values(["season_start", "team", "matchweek", "date"]).reset_index(drop=True)
    tm["rank_before"] = tm.groupby(["season_start", "team"])["rank_after"].shift(1)

    # 15) Rang adversaire (self-join)
    opp_lookup = tm[[
        "season_start", "matchweek", "team", "rank_before", "rank_after"
    ]].rename(columns={
        "team": "opponent",
        "rank_before": "opp_rank_before",
        "rank_after": "opp_rank_after"
    })

    tm = tm.merge(
        opp_lookup,
        on=["season_start", "matchweek", "opponent"],
        how="left"
    )

    return matches_clean, tm


# --- RUN ---
matches_clean, team_matches = build_team_matches(df_all)

print(matches_clean.shape, team_matches.shape)
team_matches


(4061, 11) (8122, 22)


,match_id,season_start,matchweek,date,team,opponent,goals_for,goals_against,attendance,venue,...,mw_key,cum_points_after,cum_points_before,cum_gf_after,cum_ga_after,cum_gd_after,rank_after,rank_before,opp_rank_before,opp_rank_after
0,2015_2015-08-09_Arsenal_West Ham United,2015,1,2015-08-09,Arsenal,West Ham United,0.0,2.0,59996,home,...,J1_2015,0,0.0,0.0,2.0,-2.0,19,NaN,NaN,4
1,2015_2015-08-16_Crystal Palace_Arsenal,2015,2,2015-08-16,Arsenal,Crystal Palace,2.0,1.0,24732,away,...,J2_2015,3,0.0,2.0,3.0,-1.0,11,19.0,3.0,7
2,2015_2015-08-24_Arsenal_Liverpool,2015,3,2015-08-24,Arsenal,Liverpool,0.0,0.0,60080,home,...,J3_2015,4,3.0,2.0,3.0,-1.0,9,11.0,3.0,3
3,2015_2015-08-29_Newcastle United_Arsenal,2015,4,2015-08-29,Arsenal,Newcastle United,1.0,0.0,50388,away,...,J4_2015,7,4.0,3.0,3.0,0.0,6,9.0,17.0,19
4,2015_2015-09-12_Arsenal_Stoke City,2015,5,2015-09-12,Arsenal,Stoke City,2.0,0.0,59963,home,...,J5_2015,10,7.0,5.0,3.0,2.0,4,6.0,18.0,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8117,2025_2026-01-24_Manchester City_Wolves,2025,23,2026-01-24,Wolves,Manchester City,0.0,2.0,52469,away,...,J23_2025,8,8.0,15.0,43.0,-28.0,20,20.0,2.0,2
8118,2025_2026-01-31_Wolves_Bournemouth,2025,24,2026-01-31,Wolves,Bournemouth,0.0,2.0,30161,home,...,J24_2025,8,8.0,15.0,45.0,-30.0,20,20.0,13.0,12
8119,2025_2026-02-07_Wolves_Chelsea,2025,25,2026-02-07,Wolves,Chelsea,1.0,3.0,29762,home,...,J25_2025,8,8.0,16.0,48.0,-32.0,20,20.0,5.0,5
8120,2025_2026-02-11_Nottingham Forest_Wolves,2025,26,2026-02-11,Wolves,Nottingham Forest,0.0,0.0,29921,away,...,J26_2025,9,8.0,16.0,48.0,-32.0,20,20.0,17.0,17


In [164]:
team_matches = team_matches.sort_values(
    ["season_start", "team", "matchweek", "date"]
).reset_index(drop=True)

team_matches["form5_points"] = (
    team_matches
    .groupby(["season_start", "team"])["points"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
)

team_matches["form5_gf"] = (
    team_matches
    .groupby(["season_start", "team"])["goals_for"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
)

team_matches["form5_ga"] = (
    team_matches
    .groupby(["season_start", "team"])["goals_against"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
)

team_matches["form5_gd"] = team_matches["form5_gf"] - team_matches["form5_ga"]


In [168]:
opp_form_lookup = team_matches[[
    "season_start",
    "matchweek",
    "team",
    "form5_points",
    "form5_gf",
    "form5_ga",
    "form5_gd"
]].rename(columns={
    "team": "opponent",
    "form5_points": "opp_form5_points",
    "form5_gf": "opp_form5_gf",
    "form5_ga": "opp_form5_ga",
    "form5_gd": "opp_form5_gd",
})


In [169]:
team_matches = team_matches.merge(
    opp_form_lookup,
    on=["season_start", "matchweek", "opponent"],
    how="left"
)


In [170]:
# 1) Renommer la colonne
team_matches = team_matches.rename(columns={"mw_key": "mw_id"})

# 2) Réordonner les colonnes
cols = team_matches.columns.tolist()

cols.remove("mw_id")
cols.insert(1, "mw_id")

team_matches = team_matches[cols]


In [171]:
team_matches["form5_points_diff"] = team_matches["form5_points"] - team_matches["opp_form5_points"]
team_matches["form5_gd_diff"] = team_matches["form5_gd"] - team_matches["opp_form5_gd"]
team_matches["rank_diff"] = team_matches["opp_rank_before"] - team_matches["rank_before"]


In [172]:
team_matches

,match_id,mw_id,season_start,matchweek,date,team,opponent,goals_for,goals_against,attendance,...,form5_gf,form5_ga,form5_gd,opp_form5_points,opp_form5_gf,opp_form5_ga,opp_form5_gd,form5_points_diff,form5_gd_diff,rank_diff
0,2015_2015-08-09_Arsenal_West Ham United,J1_2015,2015,1,2015-08-09,Arsenal,West Ham United,0.0,2.0,59996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015_2015-08-16_Crystal Palace_Arsenal,J2_2015,2015,2,2015-08-16,Arsenal,Crystal Palace,2.0,1.0,24732,...,0.0,2.0,-2.0,3.0,3.0,1.0,2.0,-3.0,-4.0,-16.0
2,2015_2015-08-24_Arsenal_Liverpool,J3_2015,2015,3,2015-08-24,Arsenal,Liverpool,0.0,0.0,60080,...,2.0,3.0,-1.0,6.0,2.0,0.0,2.0,-3.0,-3.0,-8.0
3,2015_2015-08-29_Newcastle United_Arsenal,J4_2015,2015,4,2015-08-29,Arsenal,Newcastle United,1.0,0.0,50388,...,2.0,3.0,-1.0,2.0,2.0,4.0,-2.0,2.0,1.0,8.0
4,2015_2015-09-12_Arsenal_Stoke City,J5_2015,2015,5,2015-09-12,Arsenal,Stoke City,2.0,0.0,59963,...,3.0,3.0,0.0,2.0,3.0,5.0,-2.0,5.0,2.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8117,2025_2026-01-24_Manchester City_Wolves,J23_2025,2025,23,2026-01-24,Wolves,Manchester City,0.0,2.0,52469,...,6.0,4.0,2.0,6.0,4.0,5.0,-1.0,0.0,3.0,-18.0
8118,2025_2026-01-31_Wolves_Bournemouth,J24_2025,2025,24,2026-01-31,Wolves,Bournemouth,0.0,2.0,30161,...,5.0,4.0,1.0,8.0,11.0,10.0,1.0,-2.0,0.0,-7.0
8119,2025_2026-02-07_Wolves_Chelsea,J25_2025,2025,25,2026-02-07,Wolves,Chelsea,1.0,3.0,29762,...,4.0,5.0,-1.0,10.0,10.0,6.0,4.0,-5.0,-5.0,-15.0
8120,2025_2026-02-11_Nottingham Forest_Wolves,J26_2025,2025,26,2026-02-11,Wolves,Nottingham Forest,0.0,0.0,29921,...,2.0,8.0,-6.0,8.0,6.0,5.0,1.0,-6.0,-7.0,-3.0


In [174]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df_model = team_matches[team_matches["goals_for"].notna()].copy()

df_model["home_dummy"] = (df_model["venue"] == "home").astype(int)

att = pd.to_numeric(df_model["attendance"], errors="coerce")
df_model["log_attendance"] = np.log(att.fillna(att.median())).astype(float)

features = ["home_dummy", "form5_gf", "opp_form5_ga", "opp_rank_before", "log_attendance"]
X = df_model[features].copy()

# forcer tout en float/int non-nullable
X = X.astype(float)

y = df_model["goals_for"].astype(float)

# enlever les lignes avec NaN (souvent J1)
mask = X.notna().all(axis=1) & y.notna()
X = sm.add_constant(X.loc[mask], has_constant="add")
y = y.loc[mask]

model_gf = sm.GLM(y, X, family=sm.families.Poisson())
res_gf = model_gf.fit()

print(res_gf.summary())


TypeError: Invalid value '32225.5' for dtype Int64

In [175]:
X.dtypes


form5_points_diff           float64
opp_form5_gf                float64
opp_form5_ga                float64
pressure_safety_18th        float64
opp_pressure_safety_18th    float64
opp_pressure_top4_4th       float64
attendance_cat_num            Int64
rivalry_intensity_num         int64
rest_days                   float64
is_home                       int64
dtype: object

In [176]:
#Vérification équipe par saison
team_per_season = (
    team_matches
    .groupby("season_start")["team"]
    .nunique()
    .reset_index(name = "nb_team")
)
team_per_season

,season_start,nb_team
0,2015,20
1,2016,20
2,2017,20
3,2018,20
4,2019,20
5,2020,20
6,2021,20
7,2022,20
8,2023,20
9,2024,20


In [177]:
#Vérification journées par saison
week_per_season = (
    team_matches
    .groupby("season_start")["matchweek"]
    .nunique()
    .reset_index(name = "nb_mw")
)

week_per_season

,season_start,nb_mw
0,2015,38
1,2016,38
2,2017,38
3,2018,38
4,2019,38
5,2020,38
6,2021,38
7,2022,38
8,2023,38
9,2024,38


In [182]:
team_matches.to_csv("fact_team_matches_bis.csv", index=False)


Fin Ingestion/Nettoyage

In [ ]:
# Sauvegarde du DataFrame pour la Partie 2
team_matches.to_csv("fact_team_matches_bis.csv", index=False)
print("✅ team_matches sauvegardé :", team_matches.shape)